# Step 2 — Train MicroSplit Model

This notebook trains a MicroSplit (μSplit) LadderVAE model on the small
cpg0000-jump-pilot dataset we built in `00_datasets.ipynb`.

For this demo we run **5 epochs** on GPU (or CPU). For production training on the
full dataset (~3500 images), see `../cpg0000-jump-pilot/train.sh` (100 epochs on HPC).

**Prerequisites:** Run `00_datasets.ipynb` and `01_noisemodels.ipynb` first.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '../../../src')  # point at JUMP-MicroSplit/src

DATASET_DIR    = Path('./cpg0000_training_dataset')
CHANNELS       = ['DNA', 'RNA', 'ER', 'AGP', 'Mito']
NOISE_MODELS   = DATASET_DIR / 'noise_models'
CHECKPOINT_DIR = DATASET_DIR / 'checkpoints'

# Training configuration (small for notebook demo)
NUM_EPOCHS      = 5
BATCH_SIZE      = 4    # small batch for limited GPU memory
TRAIN_GRID_SIZE = 32
VAL_FRACTION    = 0.2
TEST_FRACTION   = 0.1

for p, name in [(DATASET_DIR, 'Dataset'), (NOISE_MODELS, 'Noise models')]:
    if not p.exists():
        raise FileNotFoundError(f'{name} not found at {p}. Run earlier notebooks first.')

print(f'Dataset:      {DATASET_DIR}')
print(f'Noise models: {NOISE_MODELS}')
print(f'Checkpoints:  {CHECKPOINT_DIR}')

## Build datasets and data loaders

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from microsplit_reproducibility.configs.data.JUMP import get_data_configs
from microsplit_reproducibility.datasets.common import create_lazy_datasets

torch.set_float32_matmul_precision('high')

train_data_config, _, _ = get_data_configs(
    channel_idx_list=CHANNELS,
    train_grid_size=TRAIN_GRID_SIZE,
)

train_dset, val_dset, test_dset, data_stats = create_lazy_datasets(
    datapath=str(DATASET_DIR),
    channel_names=CHANNELS,
    train_grid_size=TRAIN_GRID_SIZE,
    val_grid_size=32,
    image_size=64,
    multiscale_lowres_count=train_data_config.multiscale_lowres_count,
    val_fraction=VAL_FRACTION,
    test_fraction=TEST_FRACTION,
    cache_size=32,
)

print(f'Train: {train_dset.get_num_frames()} frames  '
      f'Val: {val_dset.get_num_frames()} frames  '
      f'Test: {test_dset.get_num_frames()} frames')

# Save training stats for prediction
mean_dict, std_dict = train_dset.get_mean_std()
stats_path = DATASET_DIR / 'training_stats.npz'
np.savez(
    str(stats_path),
    mean_input  = np.array(mean_dict['input']),
    std_input   = np.array(std_dict['input']),
    mean_target = np.array(mean_dict['target']),
    std_target  = np.array(std_dict['target']),
    max_val     = np.array(train_dset.get_max_val()),
)
print(f'Training stats saved: {stats_path}')

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=0, pin_memory=False)
train_loader  = DataLoader(train_dset, shuffle=True,  **loader_kwargs)
val_loader    = DataLoader(val_dset,   shuffle=False, **loader_kwargs)

## Build model

In [ ]:
from careamics.lightning import VAEModule

from microsplit_reproducibility.configs.factory import (
    create_algorithm_config,
    get_likelihood_config,
    get_loss_config,
    get_model_config,
    get_lr_scheduler_config,
    get_optimizer_config,
    get_training_config,
)
from microsplit_reproducibility.configs.parameters.JUMP import get_microsplit_parameters

experiment_params = get_microsplit_parameters(
    nm_path=str(NOISE_MODELS),
    channel_idx_list=CHANNELS,
    batch_size=BATCH_SIZE,
)
experiment_params['data_stats'] = data_stats

loss_config         = get_loss_config(**experiment_params)
model_config        = get_model_config(**experiment_params)
gaussian_lik_config, noise_model_config, nm_lik_config = get_likelihood_config(**experiment_params)
training_config     = get_training_config(**experiment_params)
lr_scheduler_config = get_lr_scheduler_config(**experiment_params)
optimizer_config    = get_optimizer_config(**experiment_params)

training_config.num_epochs = NUM_EPOCHS

experiment_config = create_algorithm_config(
    algorithm=experiment_params['algorithm'],
    loss_config=loss_config,
    model_config=model_config,
    gaussian_lik_config=gaussian_lik_config,
    nm_config=noise_model_config,
    nm_lik_config=nm_lik_config,
    lr_scheduler_config=lr_scheduler_config,
    optimizer_config=optimizer_config,
)

model = VAEModule(algorithm_config=experiment_config)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## Train

In [ ]:
import os
from pytorch_lightning import Trainer
from microsplit_reproducibility.utils.callbacks import get_callbacks

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
callbacks = get_callbacks(str(CHECKPOINT_DIR))
for cb in callbacks:
    if hasattr(cb, 'patience'):
        cb.patience = 50

accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
print(f'Using accelerator: {accelerator}')
if accelerator == 'gpu':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

trainer = Trainer(
    max_epochs=NUM_EPOCHS,
    accelerator=accelerator,
    enable_progress_bar=True,
    callbacks=callbacks,
    precision=training_config.precision,
    gradient_clip_val=training_config.gradient_clip_val,
    gradient_clip_algorithm=training_config.gradient_clip_algorithm,
    check_val_every_n_epoch=1,
    default_root_dir=str(DATASET_DIR),
)

trainer.fit(
    model=model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

print(f'\nTraining complete. Checkpoints in: {CHECKPOINT_DIR}')

## Inspect checkpoints

In [ ]:
ckpts = sorted(CHECKPOINT_DIR.glob('*.ckpt'))
print(f'Checkpoints saved: {len(ckpts)}')
for c in ckpts:
    print(f'  {c.name}  ({c.stat().st_size // 1024 / 1024:.1f} MB)')

## Next step

Open **03_predict.ipynb** to run predictions and visualise channel unmixing.